### Fetcher Tests

#### Manual Tests
These tests print the outputs of fetch_prices and fetch_ohlcv. Compare them against Yahoo Finance's own page.

In [6]:
import sys
sys.path.append('..')

from data.fetcher import fetch_prices, fetch_ohlcv

# fetch_prices: single ticker
prices = fetch_prices('AAPL', start='2023-01-01', end='2023-06-01')
assert list(prices.columns) == ['AAPL']
assert not prices.empty
prices.head()

Ticker,AAPL
Date,
2023-01-03,122.876747
2023-01-04,124.144127
2023-01-05,122.827621
2023-01-06,127.346947
2023-01-09,127.867630


In [7]:
# fetch_prices: multiple tickers
multi = fetch_prices(['AAPL', 'MSFT'], start='2023-01-01', end='2023-06-01')
assert set(multi.columns) == {'AAPL', 'MSFT'}
assert not multi.empty
multi.head()

Ticker,AAPL,MSFT
Date,,
2023-01-03,122.876747,232.510574
2023-01-04,124.144127,222.339783
2023-01-05,122.827621,215.750122
2023-01-06,127.346947,218.292862
2023-01-09,127.867630,220.418198


In [8]:
# fetch_ohlcv: full OHLCV for one ticker
ohlcv = fetch_ohlcv('AAPL', start='2023-01-01', end='2023-06-01')
assert {'Open', 'High', 'Low', 'Close', 'Volume'} <= set(ohlcv.columns)
assert not ohlcv.empty
ohlcv.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-01-03,122.876747,128.604505,121.992528,127.995383,112117500
2023-01-04,124.144127,126.403797,122.886574,124.664832,89113600
2023-01-05,122.827621,125.529397,122.572186,124.900621,80962700
2023-01-06,127.346947,128.005196,122.699897,123.800259,87754700
2023-01-09,127.867630,131.070471,127.612195,128.182026,70790800


#### Cross-check against Yahoo's own history page

Scrapes the rendered HTML table at `finance.yahoo.com/quote/{ticker}/history` directly —
an independent path from the JSON API `yfinance` calls — and compares it to
`fetch_ohlcv(..., auto_adjust=False)`. Automated but brittle (breaks if Yahoo changes their markup).

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


def scrape_yahoo_history(ticker: str, period1: int, period2: int) -> pd.DataFrame:
    """
    Parse Yahoo Finance's own history page table (not the API yfinance calls).
    period1/period2 are unix timestamps (seconds), as used in the page's URL.
    Raises RuntimeError if the page doesn't match the expected shape -- the
    simplest signal that Yahoo has changed their markup.
    """
    url = (f'https://finance.yahoo.com/quote/{ticker}/history/'
           f'?period1={period1}&period2={period2}')
    resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, 'lxml')

    table = soup.find('table')
    if table is None:
        raise RuntimeError('no <table> found on the page -- Yahoo may have changed their markup')

    tbody = table.find('tbody')
    if tbody is None:
        raise RuntimeError('table has no <tbody> -- Yahoo may have changed their markup')

    rows = tbody.find_all('tr')
    if not rows:
        raise RuntimeError('table has no rows -- Yahoo may have changed their markup')

    cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    all_records = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in rows]
    records = [r for r in all_records if len(r) == len(cols)]  # drop dividend/split rows

    dropped = len(all_records) - len(records)
    if not records or dropped > len(all_records) * 0.5:
        raise RuntimeError(
            f'{dropped}/{len(all_records)} rows did not have the expected {len(cols)} '
            f'columns (Date/Open/High/Low/Close/Adj Close/Volume) -- dividend rows '
            f'normally account for only a handful, so this looks like a markup change')

    df = pd.DataFrame(records, columns=cols)
    try:
        df['Date'] = pd.to_datetime(df['Date'], format='%b %d, %Y')
        for c in cols[1:-1]:
            df[c] = df[c].astype(float)
        df['Volume'] = df['Volume'].str.replace(',', '').astype(int)
    except (ValueError, TypeError) as e:
        raise RuntimeError(
            f'could not parse scraped values as expected ({e}) -- '
            f'Yahoo may have changed their markup') from e

    return df.set_index('Date').sort_index()

In [10]:
# Jan 1 2023 - Jun 1 2023, same window as the URL this check is based on
scraped = scrape_yahoo_history('AAPL', period1=1672531200, period2=1685577600)
raw = fetch_ohlcv('AAPL', start='2023-01-01', end='2023-06-01', auto_adjust=False)

common = scraped.index.intersection(raw.index)
assert len(common) == len(scraped) == len(raw)

for col in ['Close', 'Adj Close', 'Volume']:
    diff = (scraped.loc[common, col] - raw.loc[common, col]).abs()
    print(f'{col}: max diff = {diff.max()}')
    assert diff.max() < 0.01, f'{col} mismatch between scraped page and yfinance'

print('yfinance matches Yahoo\'s own history page for all', len(common), 'trading days')

Close: max diff = 7.32421875682121e-06
Adj Close: max diff = 0.004940185546871589
Volume: max diff = 0
yfinance matches Yahoo's own history page for all 103 trading days
